# XTTS-v2 Partial Fine-Tuning for Stuttered Speech
This notebook trains XTTS-v2 on FluencyBank disfluent speech data using partial fine-tuning (freezing vocoder, encoder, and embeddings, training top GPT layers).

### Step 1: Check GPU and Environment
Ensure you have selected `Runtime` -> `Change runtime type` -> `T4 GPU`.

In [ ]:
!nvidia-smi

### Step 2: Auto-Accept Coqui Terms & Mount Google Drive

In [ ]:
import os
os.environ["COQUI_TOS_AGREED"] = "1"

from google.colab import drive
drive.mount('/content/drive')

### Step 3: Install Coqui TTS and Dependencies

In [ ]:
!pip install --upgrade pip
!pip install coqui-tts soundfile "transformers==4.46.3" "tokenizers==0.20.3"

### Step 4: Upload training script & Unzip Dataset
1. Upload `train_xtts_partial.py` to `/content/`.
2. Ensure `dataset_disfluent.zip` is in your Drive (`MyDrive/dataset_disfluent.zip`).

In [ ]:
!unzip -q /content/drive/MyDrive/dataset_disfluent.zip -d /content/
# Verify extraction
!ls /content/dataset_disfluent/ | head -5
!wc -l /content/dataset_disfluent/metadata_train.csv

### Step 5: Launch or Resume Partial Fine-Tuning
Uses `GPTTrainer` to train top 6 GPT layers while keeping all other components frozen.

- Checkpoints automatically save directly to Google Drive (`/content/drive/MyDrive/xtts_stutter_checkpoints`).
- **To Resume:** If your session got interrupted, pass `--continue-path` pointing to the previous run directory in Google Drive (or run the Option B cell below).

In [ ]:
# Option A: Start fresh training
!python /content/train_xtts_partial.py \
  --dataset-dir /content/dataset_disfluent \
  --output-path /content/drive/MyDrive/xtts_stutter_checkpoints \
  --unfreeze-gpt-layers 6 \
  --batch-size 2 \
  --grad-accum 84 \
  --lr 5e-6

In [ ]:
# Option B: RESUME interrupted training from where it stopped
import glob, os
run_dirs = sorted(glob.glob('/content/drive/MyDrive/xtts_stutter_checkpoints/run-*'))
latest_run = run_dirs[-1] if run_dirs else ''
print(f"Resuming from: {latest_run}")

!python /content/train_xtts_partial.py \
  --dataset-dir /content/dataset_disfluent \
  --output-path /content/drive/MyDrive/xtts_stutter_checkpoints \
  --continue-path "$latest_run" \
  --unfreeze-gpt-layers 6 \
  --batch-size 2 \
  --grad-accum 84 \
  --lr 5e-6

### Step 6: Save Key Files from Checkpoint
After training, copy the 4 essential files you need for inference.

In [ ]:
import glob, shutil, os

ckpt_root = "/content/drive/MyDrive/xtts_stutter_checkpoints"

# Find the latest run directory
run_dirs = sorted(glob.glob(os.path.join(ckpt_root, "run-*")))
if run_dirs:
    latest = run_dirs[-1]
    print(f"Latest run: {latest}")
    for f in ["best_model.pth", "config.json"]:
        src = os.path.join(latest, f)
        if os.path.isfile(src):
            print(f"  ✓ {f} ({os.path.getsize(src)/1e6:.1f} MB)")
        else:
            print(f"  ✗ {f} not found")
    # vocab.json and speakers_xtts.pth may be in the XTTS_v2.0_original_model_files subdir
    orig_dir = os.path.join(ckpt_root, "XTTS_v2.0_original_model_files")
    for f in ["vocab.json"]:
        for d in [latest, orig_dir]:
            src = os.path.join(d, f)
            if os.path.isfile(src):
                print(f"  ✓ {f} (in {d})")
                break
else:
    print("No run directories found!")

### Step 7: Test Inference with Fine-Tuned Model
Loads the base XTTS-v2 architecture and seamlessly injects your fine-tuned GPT weights.

In [ ]:
# 0. Patch gpt_inference.py to support tuple KV-cache format
import glob, os, urllib.request
py_path = "/usr/local/lib/python3.13/dist-packages/TTS/tts/layers/xtts/gpt_inference.py"
if os.path.isfile(py_path):
    with open(py_path, "r") as f:
        code = f.read()
    old_target = "past_key_values.get_seq_length()"
    safe_rep = '(past_key_values.get_seq_length() if hasattr(past_key_values, "get_seq_length") else past_key_values[0][0].shape[-2])'
    if old_target in code:
        code = code.replace(old_target, safe_rep)
        with open(py_path, "w") as f:
            f.write(code)
        print("✓ Successfully patched gpt_inference.py for KV-cache compatibility")

import torch
import soundfile as sf
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

ckpt_root = "/content/drive/MyDrive/xtts_stutter_checkpoints"
run_dirs = sorted(glob.glob(os.path.join(ckpt_root, "run-*")) + glob.glob(os.path.join(ckpt_root, "xtts_stutter_partial-*")))
latest_run = run_dirs[-1] if run_dirs else ckpt_root
orig_dir = os.path.join(ckpt_root, "XTTS_v2.0_original_model_files")
os.makedirs(orig_dir, exist_ok=True)

# 1. Ensure official base XTTS-v2 config.json is present
base_config = os.path.join(orig_dir, "config.json")
if not os.path.isfile(base_config):
    print("Downloading official XTTS-v2 config.json...")
    urllib.request.urlretrieve("https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/config.json", base_config)

# 2. Load base model architecture
print("Initializing base XTTS-v2...")
config = XttsConfig()
config.load_json(base_config)
model = Xtts.init_from_config(config)

# Load base weights and vocoder
model.load_checkpoint(
    config,
    checkpoint_dir=orig_dir,
    vocab_path=os.path.join(orig_dir, "vocab.json"),
    eval=True,
    use_deepspeed=False
)

# 3. Inject fine-tuned weights from best_model.pth (stripping 'xtts.' prefix if present)
target_path = "/content/drive/MyDrive/xtts_stutter_checkpoints/xtts_stutter_partial-November-16-2026_10+08AM-0000000/best_model.pth"
best_model_path = target_path if os.path.isfile(target_path) else os.path.join(latest_run, "best_model.pth")
print(f"Injecting fine-tuned weights from: {best_model_path}")
ckpt = torch.load(best_model_path, map_location="cpu")
state_dict = ckpt["model"] if "model" in ckpt else ckpt

clean_state = {}
for k, v in state_dict.items():
    key = k[5:] if k.startswith("xtts.") else k
    clean_state[key] = v

model.load_state_dict(clean_state, strict=False)
model.eval()
model.cuda()
print("Model loaded and ready on GPU!")

# 4. Synthesize test sentence with disfluency
test_text = "Up until last year I was a co co covert stutterer."
ref_wav = "/content/dataset_disfluent/wavs/24fb_PAR_000_29.240_34.830.wav"

print(f"Generating speech: '{test_text}'...")
outputs = model.synthesize(
    test_text,
    config,
    speaker_wav=ref_wav,
    language="en",
    temperature=0.7,
    repetition_penalty=10.0
)

out_path = "/content/stutter_test.wav"
sf.write(out_path, outputs["wav"], 24000)
print(f"SUCCESS! Audio saved at: {out_path}")

# 5. Play audio in notebook
import IPython.display as ipd
ipd.Audio(out_path)

### Step 8: Multi-Prompt Disfluency Test Suite
Tests multiple stuttering phenomena (Part-Word Repetition, Whole-Word Repetition, Prolongation, Fluent baseline) using your fine-tuned model.

In [ ]:
import IPython.display as ipd

test_cases = [
    ("part_word_rep", "I w-w-want to go to the m-m-market today."),
    ("whole_word_rep", "Up until last year I was a co co covert stutterer."),
    ("prolongation", "Sssss-sometimes it is very hard to speak."),
    ("fluent_baseline", "Up until last year I was a covert stutterer.")
]

ref_wav = "/content/dataset_disfluent/wavs/24fb_PAR_000_29.240_34.830.wav"

for label, text in test_cases:
    print(f"\nGenerating [{label}]: {text}")
    out = model.synthesize(
        text,
        config,
        speaker_wav=ref_wav,
        language="en",
        temperature=0.7,
        repetition_penalty=10.0
    )
    save_path = f"/content/test_{label}.wav"
    sf.write(save_path, out["wav"], 24000)
    print(f"  → Saved: {save_path}")
    display(ipd.Audio(save_path))

### Step 9: Package & Download Checkpoint for BTP Submission
Bundles the 4 essential files (`best_model.pth`, `config.json`, `vocab.json`, `mel_stats.pth`) into a single zip file for safe keeping.

In [ ]:
import shutil, os

export_dir = "/content/xtts_stutter_final_model"
os.makedirs(export_dir, exist_ok=True)

# Copy essential files
shutil.copy2(best_model_path, os.path.join(export_dir, "best_model.pth"))
shutil.copy2(base_config, os.path.join(export_dir, "config.json"))
shutil.copy2(os.path.join(orig_dir, "vocab.json"), os.path.join(export_dir, "vocab.json"))
shutil.copy2(os.path.join(orig_dir, "mel_stats.pth"), os.path.join(export_dir, "mel_stats.pth"))

# Create zip
zip_path = "/content/drive/MyDrive/xtts_stuttered_model_final.zip"
shutil.make_archive("/content/drive/MyDrive/xtts_stuttered_model_final", 'zip', export_dir)
print(f"✓ Packaged model ready in Drive at: {zip_path}")
!ls -lh /content/drive/MyDrive/xtts_stuttered_model_final.zip